In [ ]:
#Theta biomarker analysis

In [1]:
"""
Theta Frontal Biomarker Analysis — 2-back vs 3-back
=====================================================
هدف: بررسی اینکه آیا theta power (4-8 Hz) در الکترودهای فرانتال
(که در grid 16x16 نقشه‌برداری شده‌اند) می‌تواند نمونه‌هایی را که
مدل 2-back را اشتباه 3-back تشخیص داد از نمونه‌های درست تفکیک کند.

اگر theta power در این دو گروه معنادار فرق داشت:
  → biomarker در سیگنال خام وجود دارد اما مدل آن را نمی‌بیند
  → راه‌حل: اضافه کردن band-power به عنوان channel ورودی

ورودی‌های مورد نیاز:
  1. پوشه preprocessed data (VPxxx_eeg_windows.npy و _labels.npy و _session_ids.npy)
  2. فایل confusion matrix از نتایج A1b — OR — مستقیماً با label های پیش‌بینی‌شده

نحوه کار:
  چون فایل predictions مستقل نداریم، از یک رویکرد proxy استفاده می‌کنیم:
  برای test session یک session را بارگذاری می‌کنیم و مستقیماً
  theta power را بین کلاس 2-back و 3-back مقایسه می‌کنیم.
  سپس برای نمونه‌های misclassified (2b→3b confusion) از confusion matrix
  A1b تخمین می‌زنیم.

سیگنال EEG پس از preprocessing:
  - شکل هر window: (16, 16, 2000)  یعنی grid مکانی 16×16، 2000 نمونه زمانی (10s @ 200Hz)
  - نرمال‌شده z-score در سطح subject
  - spatial mapping: azimuthal equidistant projection
  - موقعیت فرانتال در grid 16×16: ردیف‌های بالایی (y کوچک)

نحوه اجرا:
  python theta_biomarker_analysis.py
"""

import os
import numpy as np
from scipy import signal, stats
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# CONFIG — فقط این بخش را تغییر دهید
# =============================================================================
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED_RUN_A1b_10s"

# session که به عنوان test session 1 استفاده شده (train: [2,3], test: [1])
# برای تحلیل theta بهترین session را انتخاب کنید (session 2 بیشترین accuracy داشت)
TEST_SESSION = 1          # تغییر دهید: 1، 2، یا 3

SFREQ      = 200.0        # نرخ نمونه‌برداری EEG
THETA_LOW  = 4.0          # باند theta: 4-8 Hz
THETA_HIGH = 8.0

# ناحیه فرانتال در grid 16×16
# spatial mapping azimuthal equidistant: قطب بالای سر (Fz) در مرکز نقشه
# ردیف‌های 0-5 (بالای grid) ناحیه فرانتال را پوشش می‌دهند
FRONTAL_ROWS = slice(0, 6)    # ردیف‌های بالایی grid = فرانتال
FRONTAL_COLS = slice(5, 11)   # ستون‌های مرکزی = midline frontal (Fz, FC1, FC2)

# تعداد نمونه برای مقایسه (از هر گروه)
N_SAMPLES = 100     # اگر کمتر از این موجود بود همه را استفاده می‌کند

ALPHA = 0.05        # آستانه معناداری

SUBJECTS = [f"VP{i:03d}" for i in range(1, 27)]
# =============================================================================


def bandpower_welch(signal_1d, fs, fmin, fmax):
    """
    محاسبه band power با روش Welch.
    ورودی: سیگنال یک‌بعدی (n_samples,)
    خروجی: میانگین PSD در باند [fmin, fmax]
    """
    n = len(signal_1d)
    # nperseg: یک ثانیه یا نصف سیگنال، هرکدام کمتر بود
    nperseg = min(int(fs), n // 2)
    if nperseg < 8:
        return np.nan

    freqs, psd = signal.welch(
        signal_1d, fs=fs,
        nperseg=nperseg,
        noverlap=nperseg // 2,
        window='hann'
    )
    mask = (freqs >= fmin) & (freqs <= fmax)
    if mask.sum() == 0:
        return np.nan
    return np.mean(psd[mask])


def compute_theta_for_windows(eeg_windows, frontal_rows, frontal_cols, fs,
                               fmin, fmax, max_n=None):
    """
    محاسبه theta power برای یک مجموعه از windows.

    ورودی: eeg_windows — شکل (n_windows, 16, 16, 2000)
    خروجی: آرایه (n_windows,) از میانگین theta power ناحیه فرانتال
    """
    if max_n is not None:
        idx = np.random.choice(len(eeg_windows), size=min(max_n, len(eeg_windows)),
                               replace=False)
        eeg_windows = eeg_windows[idx]

    n = len(eeg_windows)
    theta_powers = np.zeros(n)

    for i, win in enumerate(eeg_windows):
        # win: (16, 16, 2000)
        # ناحیه فرانتال: زیرمجموعه‌ای از grid
        frontal_region = win[frontal_rows, frontal_cols, :]  # (n_rows, n_cols, 2000)

        # میانگین روی همه pixel های فرانتال → سیگنال یک‌بعدی
        frontal_avg = frontal_region.mean(axis=(0, 1))  # (2000,)

        theta_powers[i] = bandpower_welch(frontal_avg, fs, fmin, fmax)

    return theta_powers


def cohen_d(a, b):
    """Cohen's d برای اندازه اثر."""
    pooled_std = np.sqrt((a.std()**2 + b.std()**2) / 2)
    return (a.mean() - b.mean()) / (pooled_std + 1e-10)


def main():
    print("=" * 65)
    print("Theta Frontal Biomarker Analysis")
    print(f"Test session: {TEST_SESSION} | Theta: {THETA_LOW}-{THETA_HIGH} Hz")
    print(f"Frontal grid region: rows {FRONTAL_ROWS}, cols {FRONTAL_COLS}")
    print("=" * 65)

    # -------------------------------------------------------------------------
    # گام ۱: بارگذاری داده‌ها
    # -------------------------------------------------------------------------
    eeg_2back_list = []
    eeg_3back_list = []
    subject_count  = 0

    for sub_id in SUBJECTS:
        sub_dir  = os.path.join(PROCESSED_DATA_PATH, sub_id)
        eeg_path = os.path.join(sub_dir, f"{sub_id}_eeg_windows.npy")
        lbl_path = os.path.join(sub_dir, f"{sub_id}_labels.npy")
        ses_path = os.path.join(sub_dir, f"{sub_id}_session_ids.npy")

        for p in [eeg_path, lbl_path, ses_path]:
            if not os.path.exists(p):
                break
        else:
            # همه فایل‌ها موجودند
            labels      = np.load(lbl_path)
            session_ids = np.load(ses_path)
            # فقط windows مربوط به test session را برمی‌داریم
            mask_session = (session_ids == TEST_SESSION)

            if mask_session.sum() == 0:
                continue

            eeg_all = np.load(eeg_path, mmap_mode='r')  # (n_windows, 16, 16, 2000)
            labels_sess = labels[mask_session]
            eeg_sess    = eeg_all[mask_session]          # فقط test session

            # تفکیک 2-back (label=1) و 3-back (label=2)
            mask_2b = (labels_sess == 1)
            mask_3b = (labels_sess == 2)

            if mask_2b.sum() > 0:
                eeg_2back_list.append(eeg_sess[mask_2b])
            if mask_3b.sum() > 0:
                eeg_3back_list.append(eeg_sess[mask_3b])

            subject_count += 1
            print(f"  ✓ {sub_id}: {mask_2b.sum()} windows 2-back, "
                  f"{mask_3b.sum()} windows 3-back (session {TEST_SESSION})")
            continue
        # اگر فایل نبود:
        print(f"  ✗ {sub_id}: فایل یافت نشد، رد شد.")

    if not eeg_2back_list or not eeg_3back_list:
        print("\nخطا: داده کافی برای تحلیل یافت نشد.")
        return

    # ترکیب همه subjects
    eeg_2back = np.concatenate(eeg_2back_list, axis=0)
    eeg_3back = np.concatenate(eeg_3back_list, axis=0)

    print(f"\nمجموع: {len(eeg_2back)} window 2-back | "
          f"{len(eeg_3back)} window 3-back از {subject_count} subject")

    # -------------------------------------------------------------------------
    # گام ۲: محاسبه theta power
    # -------------------------------------------------------------------------
    print(f"\nمحاسبه theta power ({THETA_LOW}-{THETA_HIGH} Hz) "
          f"در ناحیه فرانتال grid...")
    print("  [این ممکن است چند دقیقه طول بکشد]")

    np.random.seed(42)

    theta_2back = compute_theta_for_windows(
        eeg_2back, FRONTAL_ROWS, FRONTAL_COLS,
        SFREQ, THETA_LOW, THETA_HIGH, max_n=N_SAMPLES
    )
    theta_3back = compute_theta_for_windows(
        eeg_3back, FRONTAL_ROWS, FRONTAL_COLS,
        SFREQ, THETA_LOW, THETA_HIGH, max_n=N_SAMPLES
    )

    # حذف NaN
    theta_2back = theta_2back[~np.isnan(theta_2back)]
    theta_3back = theta_3back[~np.isnan(theta_3back)]

    if len(theta_2back) < 5 or len(theta_3back) < 5:
        print("داده کافی برای آزمون آماری وجود ندارد.")
        return

    # -------------------------------------------------------------------------
    # گام ۳: آزمون آماری
    # -------------------------------------------------------------------------
    t_stat, p_val = stats.ttest_ind(theta_2back, theta_3back, equal_var=False)
    d             = cohen_d(theta_2back, theta_3back)
    _, p_mwu      = stats.mannwhitneyu(theta_2back, theta_3back, alternative='two-sided')

    print("\n" + "=" * 65)
    print("نتایج آزمون آماری: theta power فرانتال (2-back vs 3-back)")
    print("=" * 65)
    print(f"  2-back theta power: mean={theta_2back.mean():.4e} | "
          f"std={theta_2back.std():.4e} | n={len(theta_2back)}")
    print(f"  3-back theta power: mean={theta_3back.mean():.4e} | "
          f"std={theta_3back.std():.4e} | n={len(theta_3back)}")
    print(f"\n  Welch t-test:       t={t_stat:.3f} | p={p_val:.4f}")
    print(f"  Mann-Whitney U:     p={p_mwu:.4f}")
    print(f"  Cohen's d:          {d:.3f}")

    sig = p_val < ALPHA
    direction = "3-back > 2-back" if theta_3back.mean() > theta_2back.mean() \
                else "2-back > 3-back"

    print(f"\n  {'✓ معنادار' if sig else '✗ غیرمعنادار'} (alpha={ALPHA})")
    if sig:
        print(f"  جهت: {direction}")

    # -------------------------------------------------------------------------
    # گام ۴: تحلیل تکمیلی — theta در مناطق مختلف grid
    # -------------------------------------------------------------------------
    print("\n" + "─" * 65)
    print("تحلیل تکمیلی: theta power در مناطق مختلف grid 16×16")
    print("─" * 65)

    regions = {
        "فرانتال مرکزی (Fz/FCz)":   (slice(2, 6),  slice(6, 10)),
        "فرانتال چپ (F3/FC5)":       (slice(2, 6),  slice(2, 6)),
        "فرانتال راست (F4/FC6)":     (slice(2, 6),  slice(10, 14)),
        "پریتال مرکزی (Pz/Cz)":      (slice(8, 12), slice(6, 10)),
        "اکسیپیتال (Oz)":            (slice(12, 16), slice(6, 10)),
    }

    best_region = None
    best_d      = 0.0

    for region_name, (rows, cols) in regions.items():
        t2 = compute_theta_for_windows(
            eeg_2back, rows, cols, SFREQ, THETA_LOW, THETA_HIGH,
            max_n=min(50, len(eeg_2back))
        )
        t3 = compute_theta_for_windows(
            eeg_3back, rows, cols, SFREQ, THETA_LOW, THETA_HIGH,
            max_n=min(50, len(eeg_3back))
        )
        t2 = t2[~np.isnan(t2)]
        t3 = t3[~np.isnan(t3)]

        if len(t2) < 3 or len(t3) < 3:
            continue

        _, p = stats.ttest_ind(t2, t3, equal_var=False)
        d_r  = cohen_d(t2, t3)

        flag = "★ معنادار" if p < ALPHA else "  "
        print(f"  {flag} {region_name:30s}: "
              f"2b={t2.mean():.3e}, 3b={t3.mean():.3e}, "
              f"d={d_r:.3f}, p={p:.4f}")

        if abs(d_r) > abs(best_d):
            best_d      = d_r
            best_region = region_name

    # -------------------------------------------------------------------------
    # گام ۵: نتیجه‌گیری
    # -------------------------------------------------------------------------
    print("\n" + "=" * 65)
    print("نتیجه‌گیری")
    print("=" * 65)

    if sig and abs(d) >= 0.3:
        print(f"""
✓ BIOMARKER CONFIRMED

theta power فرانتال بین 2-back و 3-back تفاوت معنادار دارد
(t={t_stat:.3f}, p={p_val:.4f}, Cohen's d={d:.3f}).

این یعنی:
  → اطلاعات لازم برای تمایز در سیگنال خام وجود دارد
  → مدل فعلی (با GAP) این اطلاعات را از دست می‌دهد
  → راه‌حل پیشنهادی: اضافه کردن theta band-power map
    به عنوان channel اضافه در ورودی EEG branch

قدم بعدی:
  preprocessing را تغییر دهید تا علاوه بر raw EEG،
  یک band-power map (16×16) برای هر باند فرکانسی محاسبه
  و به عنوان channel اضافه ذخیره شود.
  ورودی EEG: (16, 16, 2000) → (16, 16, 2000+5_bands)
  یا به صورت جداگانه: (16, 16, n_timepoints_per_band × 5)
""")
    elif sig and abs(d) < 0.3:
        print(f"""
~ BIOMARKER WEAK

تفاوت معنادار است (p={p_val:.4f}) اما اندازه اثر کوچک است (d={d:.3f}).
theta power وجود دارد اما به‌تنهایی کافی نیست.
پیشنهاد: بررسی ترکیب theta + alpha یا P300 latency.
""")
    else:
        print(f"""
✗ BIOMARKER NOT FOUND IN THIS REGION/SESSION

theta power فرانتال بین 2-back و 3-back تفاوت معنادار ندارد
(p={p_val:.4f}).

دلایل احتمالی:
  1. z-score normalization در preprocessing، amplitude را حذف کرده
     و فقط الگوی نسبی باقی مانده — theta power مطلق قابل مقایسه نیست
  2. spatial mapping 16×16 ممکن است ناحیه فرانتال را دقیق capture نکند
  3. اطلاعات theta در این dataset کافی نیست برای تمایز 2/3-back

پیشنهاد: آزمایش را روی سیگنال قبل از spatial mapping
(روی فایل fif.eeg_clean) تکرار کنید.
""")

    if best_region:
        print(f"بهترین ناحیه grid برای تمایز: {best_region} (|d|={abs(best_d):.3f})")

    # ذخیره نتایج خلاصه
    results_dir = os.path.join(os.path.dirname(PROCESSED_DATA_PATH),
                               "theta_biomarker_results")
    os.makedirs(results_dir, exist_ok=True)
    out_path = os.path.join(results_dir,
                            f"theta_results_session{TEST_SESSION}.txt")

    summary = (
        f"Theta Biomarker Analysis — Session {TEST_SESSION}\n"
        f"2-back: mean={theta_2back.mean():.4e}, std={theta_2back.std():.4e}, n={len(theta_2back)}\n"
        f"3-back: mean={theta_3back.mean():.4e}, std={theta_3back.std():.4e}, n={len(theta_3back)}\n"
        f"t={t_stat:.3f}, p={p_val:.4f}, MWU_p={p_mwu:.4f}, Cohen_d={d:.3f}\n"
        f"Significant: {sig}\n"
        f"Best region: {best_region} (|d|={abs(best_d):.3f})\n"
    )
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(summary)
    print(f"\n✓ نتایج ذخیره شد: {out_path}")

    # -------------------------------------------------------------------------
    # گام ۶: نمودار (اختیاری)
    # -------------------------------------------------------------------------
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(1, 3, figsize=(14, 5))
        fig.suptitle(
            f"Theta Power Analysis — Session {TEST_SESSION} | "
            f"2-back vs 3-back\n"
            f"t={t_stat:.3f}, p={p_val:.4f}, Cohen's d={d:.3f}",
            fontsize=11
        )

        # نمودار ۱: boxplot
        ax = axes[0]
        bp = ax.boxplot([theta_2back, theta_3back],
                        patch_artist=True, widths=0.5,
                        medianprops=dict(color='black', linewidth=2))
        bp['boxes'][0].set_facecolor('#2a78d6')
        bp['boxes'][0].set_alpha(0.7)
        bp['boxes'][1].set_facecolor('#e34948')
        bp['boxes'][1].set_alpha(0.7)
        ax.set_xticks([1, 2])
        ax.set_xticklabels(['2-back', '3-back'])
        ax.set_ylabel("Theta power (Welch PSD)")
        ax.set_title("Boxplot مقایسه")
        ax.grid(axis='y', alpha=0.3)
        sig_text = f"p={p_val:.4f} {'*' if sig else 'n.s.'}\nd={d:.3f}"
        ax.text(0.5, 0.95, sig_text, transform=ax.transAxes,
                ha='center', va='top', fontsize=9,
                color='green' if sig else 'gray')

        # نمودار ۲: histogram
        ax = axes[1]
        common_range = (
            min(theta_2back.min(), theta_3back.min()),
            max(theta_2back.max(), theta_3back.max())
        )
        bins = np.linspace(*common_range, 30)
        ax.hist(theta_2back, bins=bins, alpha=0.6, color='#2a78d6',
                label='2-back', density=True)
        ax.hist(theta_3back, bins=bins, alpha=0.6, color='#e34948',
                label='3-back', density=True)
        ax.axvline(theta_2back.mean(), color='#2a78d6', linestyle='--', lw=2)
        ax.axvline(theta_3back.mean(), color='#e34948', linestyle='--', lw=2)
        ax.set_xlabel("Theta power")
        ax.set_ylabel("چگالی")
        ax.set_title("توزیع theta power")
        ax.legend()
        ax.grid(alpha=0.3)

        # نمودار ۳: Cohen's d در مناطق مختلف
        ax = axes[2]
        region_names_short = ["Frontal\nCenter", "Frontal\nLeft",
                               "Frontal\nRight", "Parietal", "Occipital"]
        region_coords = [
            (slice(2, 6), slice(6, 10)),
            (slice(2, 6), slice(2, 6)),
            (slice(2, 6), slice(10, 14)),
            (slice(8, 12), slice(6, 10)),
            (slice(12, 16), slice(6, 10)),
        ]
        d_values = []
        for rows, cols in region_coords:
            t2 = compute_theta_for_windows(
                eeg_2back, rows, cols, SFREQ, THETA_LOW, THETA_HIGH,
                max_n=30
            )
            t3 = compute_theta_for_windows(
                eeg_3back, rows, cols, SFREQ, THETA_LOW, THETA_HIGH,
                max_n=30
            )
            t2 = t2[~np.isnan(t2)]
            t3 = t3[~np.isnan(t3)]
            d_values.append(cohen_d(t2, t3) if len(t2) > 2 and len(t3) > 2
                            else 0.0)

        colors = ['#e34948' if abs(v) >= 0.3 else '#c3c2b7' for v in d_values]
        bars = ax.barh(region_names_short, d_values, color=colors, height=0.5)
        ax.axvline(0, color='black', lw=0.5)
        ax.axvline(0.3, color='gray', lw=0.5, linestyle='--', alpha=0.5)
        ax.axvline(-0.3, color='gray', lw=0.5, linestyle='--', alpha=0.5)
        ax.set_xlabel("Cohen's d (2-back - 3-back)")
        ax.set_title("اندازه اثر در مناطق grid")
        ax.grid(axis='x', alpha=0.3)
        for bar, val in zip(bars, d_values):
            ax.text(val + 0.01 * np.sign(val + 1e-5),
                    bar.get_y() + bar.get_height() / 2,
                    f"{val:.2f}", va='center', fontsize=8)

        plt.tight_layout()
        plot_path = os.path.join(results_dir,
                                 f"theta_plot_session{TEST_SESSION}.png")
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"✓ نمودار ذخیره شد: {plot_path}")

    except ImportError:
        print("matplotlib یافت نشد — نمودار رسم نشد.")
    except Exception as e:
        print(f"خطا در رسم نمودار: {e}")


if __name__ == "__main__":
    main()

Theta Frontal Biomarker Analysis
Test session: 1 | Theta: 4.0-8.0 Hz
Frontal grid region: rows slice(0, 6, None), cols slice(5, 11, None)
  ✓ VP001: 93 windows 2-back, 93 windows 3-back (session 1)
  ✓ VP002: 93 windows 2-back, 93 windows 3-back (session 1)
  ✓ VP003: 93 windows 2-back, 93 windows 3-back (session 1)
  ✓ VP004: 67 windows 2-back, 84 windows 3-back (session 1)
  ✓ VP005: 93 windows 2-back, 93 windows 3-back (session 1)
  ✓ VP006: 93 windows 2-back, 93 windows 3-back (session 1)
  ✓ VP007: 93 windows 2-back, 93 windows 3-back (session 1)
  ✓ VP008: 93 windows 2-back, 93 windows 3-back (session 1)
  ✓ VP009: 93 windows 2-back, 93 windows 3-back (session 1)
  ✓ VP010: 93 windows 2-back, 93 windows 3-back (session 1)
  ✓ VP011: 93 windows 2-back, 93 windows 3-back (session 1)
  ✓ VP012: 93 windows 2-back, 93 windows 3-back (session 1)
  ✓ VP013: 93 windows 2-back, 83 windows 3-back (session 1)
  ✓ VP014: 93 windows 2-back, 78 windows 3-back (session 1)
  ✓ VP015: 114 windows

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 3, the array at index 0 has size 2000 and the array at index 3 has size 600